# Week 3 示例：文本表示与图像基础

- 作者：共享仓库示例
- 日期：2026-07-14
- 来源：`暑期居家集训学习计划.md` → Week 3 → NLP 文本表示基础 / CV 图像基础
- 适用周次：Week 3
- 分类：NLP / Computer Vision
- 关键词：TF-IDF、t-SNE、RGB/BGR、Canny
- 运行环境：Python 3.10+、NumPy、Matplotlib、scikit-learn、opencv-python（可选）
- 适合读者：已经会基本 Python、NumPy 和可视化的初学者

## 学习目标

1. 将文本转换为 TF-IDF 特征并进行二维可视化。
2. 理解 RGB/BGR、灰度化和边缘检测的基本流程。

> Week 3 按 NLP/CV 双轨学习，下面各保留一个最小示例，不代表完整方向作业。

本 Notebook 使用离线构造的小数据，避免第一次运行就依赖外部语料、预训练词向量或 `sample.jpg`。

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE


## 1. NLP：TF-IDF 与 t-SNE

原文要求从文本预处理、TF-IDF、Word2Vec 一直到词向量空间可视化。这里先用 TF-IDF 产生文本向量，再用 t-SNE 压到二维。

In [ ]:
def preprocess_en(text):
    text = text.lower()
    return re.sub(r'[^a-z\s]', ' ', text)

texts = [
    'image classification with convolutional networks',
    'deep learning models classify images',
    'vision features can detect objects',
    'text classification uses word representations',
    'language models learn useful text features',
    'tf idf represents documents with sparse vectors',
    'video understanding models temporal features',
    'temporal modeling is useful for video analysis',
    'anomaly detection compares normal and unusual patterns',
]
clean_texts = [preprocess_en(t) for t in texts]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
vectors = vectorizer.fit_transform(clean_texts).toarray()

positions = TSNE(
    n_components=2, perplexity=3, random_state=42,
    init='random', learning_rate='auto'
).fit_transform(vectors)

plt.figure(figsize=(8, 6))
plt.scatter(positions[:, 0], positions[:, 1], c=np.arange(len(texts)), cmap='tab10')
for i, text in enumerate(texts):
    plt.annotate(f'{i}: {text[:18]}...', positions[i], fontsize=8)
plt.title('TF-IDF Document Space with t-SNE')
plt.tight_layout()
plt.show()

## 2. CV：RGB/BGR、灰度化与边缘检测

原文使用 OpenCV 读取 `sample.jpg`。这里生成一个简单的彩色测试图，先跑通颜色空间转换、Canny 和直方图流程。

In [ ]:
try:
    import cv2
except ImportError:
    cv2 = None

if cv2 is None:
    print('未安装 opencv-python，跳过 CV 示例。')
else:
    img_rgb = np.zeros((128, 128, 3), dtype=np.uint8)
    img_rgb[20:100, 20:60] = [220, 80, 80]
    img_rgb[40:115, 70:110] = [80, 150, 220]
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, threshold1=50, threshold2=150)

    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    axes[0].imshow(img_rgb); axes[0].set_title('RGB')
    axes[1].imshow(gray, cmap='gray'); axes[1].set_title('Grayscale')
    axes[2].imshow(edges, cmap='gray'); axes[2].set_title('Canny edges')
    for ax in axes: ax.axis('off')
    plt.tight_layout()
    plt.show()

## 小结

后续可以把这份样例扩展为 Word2Vec + t-SNE、真实图像预处理、Sobel 梯度和像素直方图实验。